# FunctionGemma Banking Tool-Calling Usage

This notebook assumes the fine-tuned model is published at:

https://huggingface.co/phattrandeveloper/functiongemma-270m-function-calling

Use it as a runtime contract when wiring the model into `flutter_gemma`: it shows the exact banking tools, the expected tool-call behavior for every scenario family in this repo, and an optional Python generation loop that mirrors the Flutter flow.

The notebook is safe to run without downloading the model. Set `RUN_MODEL = True` when you want to actually load the Hugging Face model and generate outputs.


## Runtime shape to preserve

The model was trained around these invariants:

- Tool names and argument keys must stay exactly the same as `src/tools/banking.py`.
- The current user's account is always `ACC_USER`.
- `bank_name` is a short bank code such as `VCB`, `TCB`, `ACB`, `MBB`, or `BIDV`.
- Amounts are integer VND values.
- Transaction date filters use `YYYY-MM-DD`; recent-transaction periods are inclusive of today.
- Tool responses are returned to the model as `role="tool"` with content shaped like `{"name": toolName, "response": result}` in the Transformers chat-template path.
- In Flutter, map the same result to `Message.toolResponse(toolName: ..., response: ...)`.

`flutter_gemma` 0.16.x supports `ModelType.functionGemma`, `Tool`, `FunctionCallResponse`, and `Message.toolResponse`. FunctionGemma mobile use normally expects a converted `.task` or `.litertlm` artifact; a raw PyTorch Hugging Face model is what this Python demo loads.


In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from collections import Counter
from datetime import date, timedelta
from pathlib import Path
from typing import Any

MODEL_ID = "phattrandeveloper/functiongemma-270m-function-calling"
HF_MODEL_URL = f"https://huggingface.co/{MODEL_ID}"

# Keep this False for documentation/example browsing. Set True to download and run the model.
RUN_MODEL = False
LOAD_TOKENIZER_ONLY = False
MAX_NEW_TOKENS = 192
SEED = 42

SCENARIO_ORDER = [
    "recent_transactions_time_period",
    "full_bank_account_number",
    "vendor_payment",
    "missing_bank_code",
    "single_matching_beneficiary_then_transfer",
    "ambiguous_beneficiary_account_then_transfer",
    "transfer_then_offer_save_beneficiary",
]

SCENARIO_LABELS = {
    "recent_transactions_time_period": "Get account transactions for a recent time period",
    "full_bank_account_number": "Transfer to a full bank account number",
    "vendor_payment": "Transfer from vendor-style payment fields",
    "missing_bank_code": "Ask for the missing bank, then transfer",
    "single_matching_beneficiary_then_transfer": "Lookup one saved beneficiary match, then transfer",
    "ambiguous_beneficiary_account_then_transfer": "Lookup multiple beneficiary matches, disambiguate, then transfer",
    "transfer_then_offer_save_beneficiary": "Transfer, check saved beneficiaries, offer save, then add beneficiary",
}

DEFAULT_CONTEXT = {
    "user_name": "Nguyen Van An",
    "user_phone": "0912 345 678",
    "current_date": "2026-06-02",
}


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src" / "tools" / "banking.py").exists():
            return path
    raise RuntimeError("Could not find the fine-tuning-tool-calling repo root")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repo root:", REPO_ROOT)
print("Model:", HF_MODEL_URL)


In [ ]:
from common import load_config, read_jsonl
from format import assistant_tool_message, developer_message, format_sample, tool_response_message
from tools.banking import TOOL_FUNCTIONS, tool_schemas

CONFIG = load_config("config/settings.yaml")
TOOLS = tool_schemas()

raw_rows = read_jsonl("data/raw/combined/final_dataset.jsonl")
if not raw_rows:
    raise RuntimeError("Missing data/raw/combined/final_dataset.jsonl. Run scripts/generate.sh first.")

scenario_counts = Counter(row.get("scenario_id", "unknown") for row in raw_rows)
examples_by_scenario = {}
for scenario_id in SCENARIO_ORDER:
    examples_by_scenario[scenario_id] = next(
        row for row in raw_rows if row.get("scenario_id") == scenario_id
    )

formatted_examples = {
    scenario_id: format_sample(row, CONFIG)
    for scenario_id, row in examples_by_scenario.items()
}

print(f"Loaded {len(raw_rows)} raw rows")
print(json.dumps(dict(scenario_counts), ensure_ascii=False, indent=2))


## Banking tool contract

This cell reads the schema from the same Python functions used to train the model. Treat this as the source of truth for Dart `Tool(...)` declarations.


In [ ]:
def compact(text: Any, limit: int = 180) -> str:
    if not isinstance(text, str):
        text = json.dumps(text, ensure_ascii=False)
    text = " ".join(text.split())
    return text if len(text) <= limit else text[: limit - 1] + "…"


def schema_function(schema: dict[str, Any]) -> dict[str, Any]:
    return schema.get("function", schema)


def tool_contract_rows(tools: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for schema in tools:
        fn = schema_function(schema)
        parameters = fn.get("parameters", {})
        rows.append(
            {
                "name": fn.get("name"),
                "required": parameters.get("required", []),
                "properties": sorted(parameters.get("properties", {}).keys()),
                "description": compact(fn.get("description", ""), 160),
            }
        )
    return rows


for row in tool_contract_rows(TOOLS):
    print(f"- {row['name']}")
    print(f"  required: {row['required']}")
    print(f"  properties: {row['properties']}")
    print(f"  description: {row['description']}")

print("\nFull JSON schemas:\n")
print(json.dumps(TOOLS, ensure_ascii=False, indent=2))


## Flutter tool declarations

`flutter_gemma`'s `Tool` constructor accepts `name`, `description`, and a `parameters` map. The generated JSON below is shaped so you can translate it directly to Dart maps.


In [ ]:
def flutter_tool_specs(tools: list[dict[str, Any]]) -> list[dict[str, Any]]:
    specs = []
    for schema in tools:
        fn = schema_function(schema)
        specs.append(
            {
                "name": fn["name"],
                "description": fn.get("description", ""),
                "parameters": fn.get("parameters", {}),
            }
        )
    return specs


FLUTTER_TOOL_SPECS = flutter_tool_specs(TOOLS)
print(json.dumps(FLUTTER_TOOL_SPECS, ensure_ascii=False, indent=2))


## Scenario gallery

Each case below comes from the generated raw dataset. The assistant path is what the model should reproduce: text when a clarification is needed, or a tool call when the next action is executable.


In [ ]:
from IPython.display import Markdown, display


def raw_turns(sample: dict[str, Any]) -> list[dict[str, Any]]:
    if "turns" in sample:
        return sample["turns"]
    turns = [{"role": "user", "content": sample["user"]}]
    if sample.get("tool_call") is None:
        turns.append({"role": "assistant", "content": sample.get("assistant_response", "")})
    else:
        turns.append({"role": "assistant", "tool_call": sample["tool_call"]})
    return turns


def assistant_step(turn: dict[str, Any]) -> str:
    if "tool_call" in turn:
        call = turn["tool_call"]
        return f"tool_call: `{call['name']}` `{json.dumps(call.get('parameters', {}), ensure_ascii=False)}`"
    return f"text: {compact(turn.get('content', ''), 500)}"


def tool_step(turn: dict[str, Any]) -> str:
    return f"tool_result: `{turn.get('name')}` `{compact(turn.get('content', {}), 500)}`"


def scenario_trace(sample: dict[str, Any]) -> list[str]:
    trace = []
    for turn in raw_turns(sample):
        role = turn["role"]
        if role == "assistant":
            trace.append(assistant_step(turn))
        elif role == "tool":
            trace.append(tool_step(turn))
        elif role == "user" and trace:
            trace.append(f"user_followup: {compact(turn.get('content', ''), 260)}")
    return trace


def first_user_text(sample: dict[str, Any]) -> str:
    for turn in raw_turns(sample):
        if turn["role"] == "user":
            return compact(turn.get("content", ""), 600)
    return ""


def render_case_markdown(sample: dict[str, Any]) -> str:
    scenario_id = sample["scenario_id"]
    lines = [
        f"### `{scenario_id}`",
        f"**Purpose:** {SCENARIO_LABELS[scenario_id]}",
        f"**First user message:** {first_user_text(sample)}",
        "**Expected path:**",
    ]
    for idx, step in enumerate(scenario_trace(sample), start=1):
        lines.append(f"{idx}. {step}")
    return "\n".join(lines)


case_markdown = "\n\n".join(
    render_case_markdown(examples_by_scenario[scenario_id])
    for scenario_id in SCENARIO_ORDER
)
display(Markdown(case_markdown))


## Machine-readable expected cases

This is the compact payload to keep around when evaluating generated behavior or when giving another agent context.


In [ ]:
def expected_case_payload(sample: dict[str, Any]) -> dict[str, Any]:
    return {
        "scenario_id": sample["scenario_id"],
        "purpose": SCENARIO_LABELS[sample["scenario_id"]],
        "first_user_message": first_user_text(sample),
        "expected_path": scenario_trace(sample),
        "note": sample.get("note"),
    }


EXPECTED_CASES = [
    expected_case_payload(examples_by_scenario[scenario_id])
    for scenario_id in SCENARIO_ORDER
]

print(json.dumps(EXPECTED_CASES, ensure_ascii=False, indent=2))


## Prompt formatting for Transformers

The Python inference path uses the tokenizer's chat template. A generation prefix contains all messages up to the assistant turn you want the model to produce, plus `add_generation_prompt=True`.


In [ ]:
def formatted_messages_for_case(scenario_id: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    row = formatted_examples[scenario_id]
    return row["messages"], row["tools"]


def assistant_checkpoint_messages(
    scenario_id: str,
    assistant_step_index: int = 0,
) -> tuple[list[dict[str, Any]], dict[str, Any], list[dict[str, Any]]]:
    messages, tools = formatted_messages_for_case(scenario_id)
    assistant_positions = [
        idx for idx, message in enumerate(messages) if message.get("role") == "assistant"
    ]
    target_position = assistant_positions[assistant_step_index]
    return messages[:target_position], messages[target_position], tools


def print_checkpoint(scenario_id: str, assistant_step_index: int = 0) -> None:
    prefix, expected, tools = assistant_checkpoint_messages(scenario_id, assistant_step_index)
    print("Scenario:", scenario_id)
    print("Prefix messages:")
    print(json.dumps(prefix, ensure_ascii=False, indent=2))
    print("\nExpected next assistant message:")
    print(json.dumps(expected, ensure_ascii=False, indent=2))
    print("\nTool count:", len(tools))


print_checkpoint("recent_transactions_time_period", 0)


In [ ]:
tokenizer = None
model = None

if RUN_MODEL or LOAD_TOKENIZER_ONLY:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)

    if RUN_MODEL:
        device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            token=hf_token,
            torch_dtype=dtype,
        )
        model.to(device)
        model.eval()
        print(f"Loaded model on {device}")
    else:
        print("Loaded tokenizer only")
else:
    print("Skipping model load. Set RUN_MODEL=True to generate.")


In [ ]:
def render_prompt_for_generation(
    scenario_id: str,
    assistant_step_index: int = 0,
) -> str:
    if tokenizer is None:
        raise RuntimeError("Set LOAD_TOKENIZER_ONLY=True or RUN_MODEL=True first.")
    prefix, _expected, tools = assistant_checkpoint_messages(scenario_id, assistant_step_index)
    return tokenizer.apply_chat_template(
        prefix,
        tools=tools,
        add_generation_prompt=True,
        tokenize=False,
    )


if tokenizer is not None:
    prompt = render_prompt_for_generation("recent_transactions_time_period")
    print(prompt[:4000])
else:
    print("Tokenizer not loaded; prompt rendering is skipped.")


## FunctionGemma output parser for Python demos

`flutter_gemma` turns FunctionGemma function-call text into `FunctionCallResponse` for you. This parser is only for local Python smoke tests against raw generated text.


In [ ]:
START_FUNCTION_CALL = "<start_function_call>"
END_FUNCTION_CALL = "<end_function_call>"
ESCAPE = "<escape>"


def split_functiongemma_args(argument_text: str) -> list[str]:
    parts: list[str] = []
    buffer: list[str] = []
    depth = 0
    in_escape = False
    i = 0
    while i < len(argument_text):
        if argument_text.startswith(ESCAPE, i):
            in_escape = not in_escape
            buffer.append(ESCAPE)
            i += len(ESCAPE)
            continue
        char = argument_text[i]
        if not in_escape:
            if char in "[{(":
                depth += 1
            elif char in "]})" and depth > 0:
                depth -= 1
            elif char == "," and depth == 0:
                part = "".join(buffer).strip()
                if part:
                    parts.append(part)
                buffer = []
                i += 1
                continue
        buffer.append(char)
        i += 1
    part = "".join(buffer).strip()
    if part:
        parts.append(part)
    return parts


def parse_functiongemma_value(value: str) -> Any:
    value = value.strip()
    if value.startswith(ESCAPE) and value.endswith(ESCAPE):
        return value[len(ESCAPE) : -len(ESCAPE)]
    if value in {"null", "None"}:
        return None
    if value == "true":
        return True
    if value == "false":
        return False
    if re.fullmatch(r"-?\d+", value):
        return int(value)
    if re.fullmatch(r"-?\d+\.\d+", value):
        return float(value)
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return value.strip('"')


def parse_functiongemma_args(argument_text: str) -> dict[str, Any]:
    args: dict[str, Any] = {}
    for part in split_functiongemma_args(argument_text):
        if ":" not in part:
            continue
        key, value = part.split(":", 1)
        args[key.strip()] = parse_functiongemma_value(value)
    return args


def clean_generated_text(text: str) -> str:
    for token in ["<eos>", "<end_of_turn>", "<bos>"]:
        text = text.replace(token, "")
    return text.strip()


def parse_generated_action(text: str) -> dict[str, Any]:
    cleaned = clean_generated_text(text)
    pattern = re.compile(
        re.escape(START_FUNCTION_CALL)
        + r"\s*call:(?P<name>[A-Za-z_]\w*)\{(?P<args>[\s\S]*?)\}\s*"
        + re.escape(END_FUNCTION_CALL)
    )
    match = pattern.search(cleaned)
    if not match:
        return {"type": "text", "content": cleaned, "raw": text}
    return {
        "type": "tool_call",
        "name": match.group("name"),
        "arguments": parse_functiongemma_args(match.group("args")),
        "raw": text,
    }


sample_generated = "<start_function_call>call:initiate_transfer{from_account:<escape>ACC_USER<escape>,to_account:<escape>1023456789<escape>,bank_name:<escape>VCB<escape>,amount:1200000,message:<escape>tiền cafe<escape>}<end_function_call>"
print(json.dumps(parse_generated_action(sample_generated), ensure_ascii=False, indent=2))


## Optional Python generation helpers

These helpers mirror the runtime loop:

1. Render messages plus tools.
2. Generate the next assistant output.
3. Parse a tool call or collect text.
4. Execute the native banking function.
5. Append the tool response and generate the next assistant output.


In [ ]:
def generate_assistant_text(
    messages: list[dict[str, Any]],
    tools: list[dict[str, Any]],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    if tokenizer is None or model is None:
        raise RuntimeError("Set RUN_MODEL=True and run the model-loading cell first.")
    import torch

    prompt = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)


def generate_for_case(scenario_id: str, assistant_step_index: int = 0) -> dict[str, Any]:
    prefix, expected, tools = assistant_checkpoint_messages(scenario_id, assistant_step_index)
    generated_text = generate_assistant_text(prefix, tools)
    return {
        "scenario_id": scenario_id,
        "assistant_step_index": assistant_step_index,
        "generated_text": generated_text,
        "parsed": parse_generated_action(generated_text),
        "expected": expected,
    }


if RUN_MODEL:
    result = generate_for_case("recent_transactions_time_period")
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("Generation helper defined. Set RUN_MODEL=True to run it.")


## Demo banking tool dispatcher

Replace these stubs with your app's real banking APIs. The important part is that the dispatch names and argument keys match the training tools exactly.


In [ ]:
def demo_get_account_info(
    account_id: str,
    info_type: str,
    from_date: str | None = None,
    to_date: str | None = None,
    limit: int = 10,
) -> dict[str, Any]:
    if info_type == "balance":
        return {"status": "success", "account_id": account_id, "balance": 12500000, "currency": "VND"}
    transactions = [
        {"date": "2026-06-02", "amount": -250000, "account": "1023456789", "description": "Chuyen khoan tien cafe"},
        {"date": "2026-05-31", "amount": 500000, "account": "0909123456", "description": "Nhan tien hoan ung"},
        {"date": "2026-05-28", "amount": -125000, "account": "6688990011", "description": "Thanh toan QR"},
    ]
    return {"status": "success", "account_id": account_id, "transactions": transactions[:limit]}


def demo_get_beneficiary_info() -> dict[str, Any]:
    return {
        "beneficiaries": [
            {"contact_name": "Nguyễn Thị Lan", "to_account": "0912233445", "bank_name": "TCB"},
            {"contact_name": "Nguyễn Thị Mai", "to_account": "0911222333", "bank_name": "TCB"},
            {"contact_name": "Trần Thị Mai", "to_account": "0944555666", "bank_name": "TCB"},
            {"contact_name": "Trần Thị Lan", "to_account": "0987654321", "bank_name": "ACB"},
        ]
    }


def demo_initiate_transfer(
    from_account: str,
    to_account: str,
    bank_name: str,
    amount: int,
    message: str | None = None,
    beneficiary_name: str | None = None,
) -> dict[str, Any]:
    return {
        "status": "success",
        "from_account": from_account,
        "to_account": to_account,
        "bank_name": bank_name,
        "amount": amount,
        "message": message,
        "beneficiary_name": beneficiary_name,
    }


def demo_add_beneficiary(contact_name: str, to_account: str, bank_name: str) -> dict[str, Any]:
    return {
        "status": "success",
        "contact_name": contact_name,
        "to_account": to_account,
        "bank_name": bank_name,
    }


TOOL_DISPATCH = {
    "get_account_info": demo_get_account_info,
    "get_beneficiary_info": demo_get_beneficiary_info,
    "initiate_transfer": demo_initiate_transfer,
    "add_beneficiary": demo_add_beneficiary,
}


def execute_tool_call(name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    if name not in TOOL_DISPATCH:
        return {"status": "error", "message": f"Unknown tool: {name}"}
    return TOOL_DISPATCH[name](**arguments)


print(execute_tool_call("get_account_info", {"account_id": "ACC_USER", "info_type": "transactions", "limit": 3}))


In [ ]:
def messages_for_user_text(user_text: str, context: dict[str, str] | None = None) -> list[dict[str, Any]]:
    context = context or DEFAULT_CONTEXT
    return [
        developer_message(context, CONFIG),
        {"role": "user", "content": user_text},
    ]


def append_assistant_tool_call(
    messages: list[dict[str, Any]],
    name: str,
    arguments: dict[str, Any],
) -> None:
    messages.append(
        {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "type": "function",
                    "function": {"name": name, "arguments": arguments},
                }
            ],
        }
    )


def append_tool_response(
    messages: list[dict[str, Any]],
    name: str,
    response: dict[str, Any],
) -> None:
    messages.append({"role": "tool", "content": {"name": name, "response": response}})


def run_single_tool_loop(user_text: str) -> dict[str, Any]:
    messages = messages_for_user_text(user_text)
    first_raw = generate_assistant_text(messages, TOOLS)
    first_action = parse_generated_action(first_raw)
    result = {"user_text": user_text, "first_raw": first_raw, "first_action": first_action}

    if first_action["type"] != "tool_call":
        result["final_text"] = first_action["content"]
        return result

    tool_result = execute_tool_call(first_action["name"], first_action["arguments"])
    append_assistant_tool_call(messages, first_action["name"], first_action["arguments"])
    append_tool_response(messages, first_action["name"], tool_result)
    final_raw = generate_assistant_text(messages, TOOLS)
    result["tool_result"] = tool_result
    result["final_raw"] = final_raw
    result["final_action"] = parse_generated_action(final_raw)
    return result


if RUN_MODEL:
    demo = run_single_tool_loop("Cho mình xem lịch sử giao dịch 7 ngày gần đây")
    print(json.dumps(demo, ensure_ascii=False, indent=2))
else:
    print("Single-tool loop defined. Set RUN_MODEL=True to run it.")


## Flutter Gemma integration sketch

The Flutter side should keep the same loop, but `flutter_gemma` handles FunctionGemma parsing and surfaces a structured `FunctionCallResponse`.

```dart
import 'package:flutter_gemma/core/api/flutter_gemma.dart';
import 'package:flutter_gemma/flutter_gemma.dart';

const modelUrl = 'https://huggingface.co/phattrandeveloper/functiongemma-270m-function-calling/resolve/main/<converted-model-file>.task';

Future<void> installBankingModel(String? hfToken) async {
  FlutterGemma.initialize(huggingFaceToken: hfToken);

  await FlutterGemma.installModel(
    modelType: ModelType.functionGemma,
  )
      .fromNetwork(modelUrl, token: hfToken)
      .withProgress((progress) => print('download: $progress%'))
      .install();
}

Future<void> handleBankingMessage(String userText) async {
  final model = await FlutterGemma.getActiveModel(
    maxTokens: 2048,
    preferredBackend: PreferredBackend.gpu,
  );

  final chat = await model.createChat(
    temperature: 0.0,
    randomSeed: 42,
    topK: 1,
    modelType: ModelType.functionGemma,
    supportsFunctionCalls: true,
    tools: bankingTools,
    systemInstruction: bankingSystemInstruction,
  );

  await chat.addQueryChunk(Message.text(text: userText, isUser: true));

  await for (final response in chat.generateChatResponseAsync()) {
    if (response is TextResponse) {
      appendTextToUi(response.token);
    } else if (response is FunctionCallResponse) {
      final toolResult = await dispatchBankingTool(response.name, response.args);
      await chat.addQueryChunk(
        Message.toolResponse(toolName: response.name, response: toolResult),
      );

      await for (final followup in chat.generateChatResponseAsync()) {
        if (followup is TextResponse) {
          appendTextToUi(followup.token);
        } else if (followup is FunctionCallResponse) {
          final nextResult = await dispatchBankingTool(followup.name, followup.args);
          await chat.addQueryChunk(
            Message.toolResponse(toolName: followup.name, response: nextResult),
          );
        }
      }
    }
  }
}
```

For production, make the follow-up handling a loop with a max tool-call count, for example 4, so multi-turn chains like `transfer_then_offer_save_beneficiary` can call `initiate_transfer`, then `get_beneficiary_info`, then `add_beneficiary` after the user confirms.


## Scenario-specific Flutter expectations

Use this checklist when testing `flutter_gemma` with the converted model:

- `recent_transactions_time_period`: first response should be `get_account_info` with `info_type="transactions"`, `from_date`, `to_date`, and `limit`; periods from 1 to 30 days are valid.
- `full_bank_account_number`: first response should be `initiate_transfer`; normalize bank names to short codes and amounts to integer VND.
- `vendor_payment`: first response should be `initiate_transfer`; parse field-style text like bank, account, content, and amount.
- `missing_bank_code`: first response should be text asking for the bank; after the user provides it, call `initiate_transfer`.
- `single_matching_beneficiary_then_transfer`: first call `get_beneficiary_info`; if exactly one beneficiary matches the name/bank, call `initiate_transfer` without asking again.
- `ambiguous_beneficiary_account_then_transfer`: first call `get_beneficiary_info`; if multiple beneficiaries match, return the disambiguation text and candidate accounts; after selection, call `initiate_transfer`.
- `transfer_then_offer_save_beneficiary`: call `initiate_transfer`, then `get_beneficiary_info`; if not already saved, ask whether to save; on confirmation, call `add_beneficiary`.


In [ ]:
def system_instruction_for_flutter(context: dict[str, str] | None = None) -> str:
    context = context or DEFAULT_CONTEXT
    return CONFIG["formatting"]["developer_prompt"].format(**context)


print(system_instruction_for_flutter())
